# 03 · Detektor bola mati (`D`)

**Tujuan.** Untuk setiap frame yang dinilai, putuskan **bola hidup atau mati**. Tahap ini tidak peduli siapa yang
menguasai bola; itu urusan tahap H/A. Keluarannya juga menandai setiap **restart** (akhir segmen `D`) beserta
tim yang kemungkinan besar melakukannya.

| | |
|---|---|
| **Input** | `cache/prep.npz`, `cache/tags.csv` |
| **Output** | `cache/dead.npz`, `cache/dead_segments_v3.csv`, `cache/restarts_v3.csv`, `cache/dead_meta.json`, laporan `cache/reports/03_bola_mati.*` |
| **Waktu** | ± 1 menit |

### Pendekatan

1. **Fitur kolektif yang halus dalam waktu** — semua dibuat sehingga "lebih besar = lebih mirip bola mati",
   lalu distandarkan (rata-rata 0, simpangan 1 di frame dinilai).
2. **Skor** = jumlah berbobot fitur + **bias**. Bobot ditetapkan dari prinsip (bukan dilatih di GT publik — 4 segmen
   terlalu sedikit; percobaan sebelumnya menunjukkan bobot terlatih kalah dari bobot manual pada validasi
   *leave-one-segment-out*).
3. **Viterbi 2 keadaan** (hidup/mati) dengan biaya setiap pergantian → segmen utuh, bukan keputusan per frame
   yang berkedip. Ingat prinsip spellF1: `D` palsu di tengah possession memecahnya jadi dua.
4. **Aturan pasti** yang memaksa `D`: babak 2 sebelum kick-off (FAQ.md), dan jeda sebelum kick-off setelah gol.
5. **Bias dipilih agar porsi `D` sesuai prior dari pengetahuan sepak bola** (waktu bermain efektif), bukan dari GT.

### Mengapa bias perlu prior?

Dengan fitur terstandar dan bias 0, Viterbi memotong kira-kira di "rata-rata frame" — porsi `D` yang dihasilkan
adalah kebetulan dari standardisasi, bukan bukti. Lebih jujur menyatakan asumsinya secara eksplisit:
waktu bermain efektif di LaLiga umumnya sekitar **50–56 menit** dari ± 95–100 menit pertandingan. Frame dinilai
± 105,9 menit (termasuk 2 menit sebelum kick-off babak 2), sehingga porsi `D` yang wajar ± **0,45–0,52**.
Prior ini *parameter yang dinyatakan*, dan bagian 6 menunjukkan seberapa sensitif hasil terhadapnya.

In [ ]:
%run ./common.ipynb
import warnings

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `WEIGHTS` | lihat sel | urutan kekuatan bukti dari prinsip + temuan 02: pemain tenang paling kuat; celah bola panjang dan bola dekat garis sedang; bola pelan dan jendela set piece lemah |
| `SWITCH_COST` | 20 | biaya (dalam satuan skor) setiap pergantian hidup↔mati. Segmen harus "dibayar" oleh bukti ± 20 satuan |
| `MIN_SEG` | 50 frame (2 s) | segmen `D` lebih pendek dari ini dibuang (bola mati di sepak bola hampir selalu > 2 s) |
| `D_SHARE_PRIOR` | 0,48 | porsi `D` sasaran di frame dinilai (lihat atas) |
| `GAP_LONG_S` | 3 s | celah bola lebih pendek dari ini dianggap biasa (oklusi singkat), tidak menambah bukti |

In [ ]:
WEIGHTS = {"calm": 1.0, "gap_long": 0.5, "near_line": 0.5, "ball_slow": 0.4, "in_sp": 0.3}
SWITCH_COST = 20.0
MIN_SEG = 50
D_SHARE_PRIOR = 0.48
GAP_LONG_S = 3.0

In [ ]:
rep = Report("03_bola_mati")
CHECKS = []


def check(name, ok, detail="", critical=True):
    ok = bool(ok)
    status = "PASS" if ok else ("FAIL" if critical else "WARN")
    CHECKS.append({"cek": name, "status": status, "kritis": critical, "keterangan": detail})
    rep("[{}] {}{}".format(status, name, " — " + detail if detail else ""))
    return ok


z = load_prep()
tags = load_tags()
P, HOME = z["players"], z["slot_is_home"]
obs = z["ball_src"] > 0
bx, by = z["ball_filled"][:, 0], z["ball_filled"][:, 1]

## 1. Fitur

| Fitur | Definisi | Mengapa |
|---|---|---|
| `calm` | −(log kecepatan pemain rata-rata ± 3 s − log median lokal ± 2 menit) | pemain melambat saat bola mati; dibanding norma lokal agar kelelahan/tempo tidak menipu (AUC 02: ± 0,84) |
| `gap_long` | log(1 + panjang celah dalam detik) untuk frame di dalam celah ≥ `GAP_LONG_S`, selain itu 0 | bola hidup hampir selalu terlihat; celah panjang mengarah ke bola mati (temuan 02 #2) |
| `near_line` | −min jarak bola ke garis lapangan dalam ± 2 s (dibatasi −5…20 m; bola hilang → 20 m) | restart terjadi di garis: lemparan, sepak pojok, tendangan gawang (± 0,77) |
| `ball_slow` | −log(1 + median kecepatan bola ± 2 s) | bola diletakkan/diam sebelum restart (± 0,71); netral bila bola hilang |
| `in_sp` | frame di dalam jendela tag `SET PIECES` | petunjuk lemah: jendela tag berdurasi tetap, bukan durasi bola mati |

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    ps = np.nanmean(P[..., 2], axis=1)
sm = pd.Series(ps).rolling(151, center=True, min_periods=50).mean()
loc = sm.rolling(6001, center=True, min_periods=500).median()
calm = -(np.log(sm + 0.1) - np.log(loc + 0.1)).to_numpy()

gap_len = np.zeros(N_FRAMES)
for h, (a, b) in HALVES.items():
    s, e = runs(~obs[a:b + 1])
    for si, ei in zip(s, e):
        gap_len[a + si:a + ei] = (ei - si) / FPS
gap_long = np.where(gap_len >= GAP_LONG_S, np.log1p(gap_len), 0.0)

line = np.where(obs, np.minimum.reduce([bx, PITCH_L - bx, by, PITCH_W - by]), np.nan)
mn = pd.Series(line).rolling(101, center=True, min_periods=10).min().to_numpy()
near_line = -np.clip(np.nan_to_num(mn, nan=20.0), -5.0, 20.0)

h = 12
ball_v = np.full(N_FRAMES, np.nan)
ball_v[h:-h] = np.hypot(bx[2 * h:] - bx[:-2 * h], by[2 * h:] - by[:-2 * h]) / (2 * h / FPS)
ball_slow = -np.log1p(pd.Series(ball_v).rolling(101, center=True, min_periods=20).median().to_numpy())

sp = tags[tags.code == "SET PIECES"]
in_sp = np.zeros(N_FRAMES)
for r in sp.itertuples():
    in_sp[max(r.start_f, 0):min(r.end_f, N_FRAMES)] = 1.0

FEATS = list(WEIGHTS)
F = np.stack([dict(calm=calm, gap_long=gap_long, near_line=near_line, ball_slow=ball_slow, in_sp=in_sp)[k]
              for k in FEATS], axis=1)
mu = np.nanmean(F[GRADED], axis=0)
sd = np.nanstd(F[GRADED], axis=0) + 1e-9
Zf = (F - mu) / sd
Zf[~np.isfinite(Zf)] = 0.0          # fitur tak terukur = netral
W = np.array([WEIGHTS[k] for k in FEATS])
base_score = Zf @ W
rep("fitur: {} | bobot: {}".format(FEATS, W.tolist()))
rep("frame dalam celah bola ≥ {:.0f} s: {:.1f}% dari frame dinilai".format(
    GAP_LONG_S, 100 * (gap_len[GRADED] >= GAP_LONG_S).mean()))

## 2. Aturan pasti

1. **Babak 2 sebelum kick-off** (`dead_prior`, 2.934 frame) — FAQ.md.
2. **Sebelum kick-off setelah gol.** Tag `Kick Off (After Goal)` berupa jendela 5 s. Untuk dua kick-off awal
   babak, waktu kick-off sebenarnya **dipublikasikan** (11.185 dan 113.742), sehingga jarak "awal tag → kick-off"
   bisa diukur tanpa GT. Jarak rata-rata itu dipakai untuk memperkirakan kick-off setelah gol, dan frame dari awal
   tag sampai perkiraan kick-off dipaksa `D`. Aturannya sengaja konservatif (hanya beberapa detik); selebrasi
   panjang sebelumnya diserahkan pada fitur.

In [ ]:
ko_tags = sp[sp.Type.str.lower().str.startswith("kick off")]
start_tags = ko_tags[ko_tags.Type.str.lower() == "kick off (start)"].sort_values("start_f")
offsets = [KICKOFF[h] - int(r.start_f) for h, r in zip((1, 2), start_tags.itertuples())]
KO_OFFSET = int(round(np.mean(offsets)))
rep("kick-off awal babak: awal tag → kick-off = {} frame → dipakai {} frame ({:.1f} s)".format(
    offsets, KO_OFFSET, KO_OFFSET / FPS))

forced = z["dead_prior"].copy()
goal_rows = []
for r in ko_tags[ko_tags.Type.str.lower() == "kick off (after goal)"].itertuples():
    ko_est = int(r.start_f) + KO_OFFSET
    forced[int(r.start_f):ko_est] = True
    goals_before = tags[(tags.code == "GOALS") & (tags.start_f < r.start_f)]
    last_goal = goals_before.start_f.max() if len(goals_before) else np.nan
    goal_rows.append({"kick-off (perkiraan)": ko_est, "babak": int(HALF[ko_est]),
                      "jarak dari tag GOALS terakhir (s)": round((ko_est - last_goal) / FPS, 1)})
goal_df = pd.DataFrame(goal_rows)
rep.table(goal_df, "kick-off setelah gol:")
forced &= GRADED
rep("frame dipaksa D: {} (prior babak 2 {} + setelah gol {})".format(
    int(forced.sum()), int(z["dead_prior"].sum()), int(forced.sum() - z["dead_prior"].sum())))

## 3. Decoder: Viterbi 2 keadaan + bias dari prior

Untuk satu nilai bias *b*: skor = `base_score + b`; emisi hidup = −skor/2, mati = +skor/2; frame dipaksa diberi
emisi ±10⁴. Viterbi dijalankan per babak (tidak melintasi jeda turun minum), lalu segmen `D` < `MIN_SEG` yang
tidak dipaksa dibuang.

Bias dicari dengan **bisection** sehingga porsi `D` di frame dinilai = `D_SHARE_PRIOR`. Porsi `D` naik monoton
terhadap *b*, jadi bisection selalu konvergen.

In [ ]:
TRANS = np.array([[0.0, SWITCH_COST], [SWITCH_COST, 0.0]])


def decode(bias):
    s = base_score + bias
    emis = np.stack([-s / 2, s / 2], axis=1)
    emis[forced, 0], emis[forced, 1] = -1e4, 1e4
    mask = np.zeros(N_FRAMES, dtype=bool)
    for h, (a, b) in HALVES.items():
        mask[a:b + 1] = viterbi(emis[a:b + 1], TRANS) == 1
    starts, ends = runs(mask)
    for a, b in zip(starts, ends):
        if b - a < MIN_SEG and not forced[a:b].any():
            mask[a:b] = False
    return mask


def d_share(mask):
    return float(mask[GRADED].mean())


with timer("bias 0 (referensi)"):
    share_b0 = d_share(decode(0.0))
rep("porsi D dengan bias 0: {:.3f}".format(share_b0))

lo, hi = -6.0, 6.0
with timer("bisection bias"):
    for _ in range(18):
        mid = (lo + hi) / 2
        if d_share(decode(mid)) < D_SHARE_PRIOR:
            lo = mid
        else:
            hi = mid
BIAS = round((lo + hi) / 2, 4)
d_mask = decode(BIAS)
rep("bias terpilih {:+.4f} → porsi D {:.3f} (sasaran {:.2f})".format(BIAS, d_share(d_mask), D_SHARE_PRIOR))
check("porsi D mengenai prior (±0,01)", abs(d_share(d_mask) - D_SHARE_PRIOR) <= 0.01, "{:.3f}".format(d_share(d_mask)))
check("semua frame dipaksa = D", d_mask[forced].all())
check("D hanya di frame dinilai", not d_mask[~GRADED].any())

## 4. Segmen `D` dan restart

Setiap segmen `D` berakhir di sebuah **restart**. Untuk setiap restart dicatat:

- jenis & tim tag `SET PIECES` terdekat (bila ada),
- **tim terdekat ke bola di 2 s terakhir segmen `D`** — temuan 02 #9: tim yang melakukan restart adalah tim yang
  berdiri di dekat bola. Bila bola tak terlihat di sana, dipakai 2 s pertama setelah restart; bila tetap tak
  terlihat → `NA` (tahap H/A yang memutuskan).

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    dH = np.where(obs, np.nanmin(np.hypot(P[:, HOME, 0] - bx[:, None], P[:, HOME, 1] - by[:, None]), axis=1), np.nan)
    dA = np.where(obs, np.nanmin(np.hypot(P[:, ~HOME, 0] - bx[:, None], P[:, ~HOME, 1] - by[:, None]), axis=1), np.nan)
near = np.where(np.isfinite(dH) & np.isfinite(dA), np.where(dH < dA, 1, -1), 0)


def nearest_team(a, b, min_visible=10):
    nn = near[max(a, 0):b]
    vis = nn != 0
    if vis.sum() < min_visible:
        return "NA", int(vis.sum())
    return ("H" if (nn[vis] == 1).mean() >= 0.5 else "A"), int(vis.sum())


seg_rows = []
for i, (a, b) in enumerate(zip(*runs(d_mask))):
    near_sp = sp[(sp.start_f < b + 300) & (sp.end_f > a)]
    tag_type, tag_side = "NA", "NA"
    if len(near_sp):
        row = near_sp.iloc[(near_sp.start_f - a).abs().argmin()]
        tag_type, tag_side = str(row.Type), str(row.side)
    team, n_vis = nearest_team(b - 50, b)
    source = "akhir D"
    if team == "NA":
        team, n_vis = nearest_team(b, b + 50)
        source = "awal hidup" if team != "NA" else "-"
    seg_rows.append({"seg": i, "half": int(HALF[a]), "start": int(a), "end_excl": int(b), "length": int(b - a),
                     "forced": bool(forced[a:b].any()), "tag_type": tag_type, "tag_side": tag_side,
                     "restart_team": team, "restart_source": source})
segs = pd.DataFrame(seg_rows)
L = segs.length[~segs.forced]
rep("segmen D: {} | median {:.1f} s | p10 {:.1f} s | p90 {:.1f} s | maks {:.1f} s".format(
    len(segs), L.median() / FPS, L.quantile(0.1) / FPS, L.quantile(0.9) / FPS, L.max() / FPS))
rep("per babak: {}".format(segs.half.value_counts().sort_index().to_dict()))
rep("tim restart: {} | sumber: {}".format(segs.restart_team.value_counts().to_dict(),
                                          segs.restart_source.value_counts().to_dict()))
both = segs[(segs.tag_side.isin(["H", "A"])) & (segs.restart_team.isin(["H", "A"]))]
rep("restart dengan tag bertim & tim terdekat terukur: {} | sepakat {}".format(
    len(both), int((both.tag_side == both.restart_team).sum())))
sp_nk = sp[~sp.Type.str.lower().str.startswith("kick off")]
hit = sum(bool(d_mask[max(r.start_f, 0):min(r.end_f, N_FRAMES)].any()) for r in sp_nk.itertuples())
rep("jendela tag set piece (bukan kick-off) yang memuat D: {} / {}".format(hit, len(sp_nk)))

## 5. Cek kebenaran pada GT publik

**Bukan** penilaian kualitas untuk memilih parameter (4 segmen). Bobot tidak dilatih di sini, jadi angka ini
adalah uji di luar sampel — tetapi tetap sangat kecil.

- presisi / recall per frame untuk `D` di jendela publik,
- jumlah segmen `D` prediksi vs GT,
- IoU terbaik setiap segmen `D` GT.

In [ ]:
gt = pm.read_label_csv(str(GT_PUBLIC))["label"].astype(str).str.strip().str.upper().to_numpy()
known = np.flatnonzero(gt != "X")
W0, W1 = int(known[0]), int(known[-1]) + 1


def window_metrics(mask):
    p, g = mask[W0:W1], gt[W0:W1] == "D"
    tp = int((p & g).sum())
    ps, pe = runs(p)
    ious = []
    for a, b in zip(*runs(g)):
        best = 0.0
        for c, d in zip(ps, pe):
            inter = max(0, min(b, d) - max(a, c))
            best = max(best, inter / float((b - a) + (d - c) - inter))
        ious.append(round(best, 2))
    return {"presisi": round(tp / max(int(p.sum()), 1), 2), "recall": round(tp / max(int(g.sum()), 1), 2),
            "segmen prediksi": len(ps), "segmen GT": len(ious), "IoU per segmen GT": ious}


pub = window_metrics(d_mask)
rep("GT publik: {}".format(pub))

## 6. Sensitivitas terhadap prior dan biaya pergantian

Bila hasil berubah drastis saat parameter digeser sedikit, parameter itu rapuh. Tabel ini **tidak** dipakai
untuk memilih nilai; ia menunjukkan seberapa besar keputusan bergantung padanya. Kolom publik hanya informasi.

In [ ]:
sens = []
for prior in (0.42, 0.45, 0.48, 0.51, 0.54):
    lo_, hi_ = -6.0, 6.0
    for _ in range(14):
        mid = (lo_ + hi_) / 2
        lo_, hi_ = (mid, hi_) if d_share(decode(mid)) < prior else (lo_, mid)
    m = decode((lo_ + hi_) / 2)
    w = window_metrics(m)
    sens.append({"prior D": prior, "bias": round((lo_ + hi_) / 2, 2), "segmen D": len(runs(m)[0]),
                 "median D (s)": round(float(np.median(np.diff(np.stack(runs(m)), axis=0))) / FPS, 1),
                 "publik presisi": w["presisi"], "publik recall": w["recall"], "publik segmen": w["segmen prediksi"]})
sens = pd.DataFrame(sens).set_index("prior D")
rep.table(sens, "sensitivitas terhadap D_SHARE_PRIOR (SWITCH_COST tetap):")

sens_c = []
for cost in (10.0, 20.0, 40.0):
    TRANS[:] = [[0.0, cost], [cost, 0.0]]
    m = decode(BIAS)
    sens_c.append({"switch cost": cost, "porsi D": round(d_share(m), 3), "segmen D": len(runs(m)[0])})
TRANS[:] = [[0.0, SWITCH_COST], [SWITCH_COST, 0.0]]
rep.table(pd.DataFrame(sens_c).set_index("switch cost"), "sensitivitas terhadap SWITCH_COST (bias tetap):")

## 7. Simpan

In [ ]:
np.savez(CACHE / "dead.npz", d_score=(base_score + BIAS).astype(np.float32), d_mask=d_mask, forced=forced,
         near_team=near.astype(np.int8))
segs.to_csv(CACHE / "dead_segments_v3.csv", index=False)
segs[["seg", "end_excl", "tag_type", "tag_side", "restart_team", "restart_source"]].rename(
    columns={"end_excl": "restart_frame"}).to_csv(CACHE / "restarts_v3.csv", index=False)
(CACHE / "dead_meta.json").write_text(json.dumps({
    "features": FEATS, "weights": WEIGHTS, "mu": mu.tolist(), "sd": sd.tolist(), "bias": BIAS,
    "switch_cost": SWITCH_COST, "min_seg": MIN_SEG, "d_share_prior": D_SHARE_PRIOR, "gap_long_s": GAP_LONG_S,
    "ko_offset": KO_OFFSET, "d_share": d_share(d_mask), "n_segments": len(segs)}, indent=2), encoding="utf-8")

summary_df = pd.DataFrame(CHECKS)
n_fail = int((summary_df.status == "FAIL").sum())
rep("\nPASS {} | WARN {} | FAIL {}".format(int((summary_df.status == "PASS").sum()),
                                         int((summary_df.status == "WARN").sum()), n_fail))
rep.save({"checks": CHECKS, "bias": BIAS, "d_share": d_share(d_mask), "share_bias0": share_b0,
          "n_segments": len(segs), "public": pub, "sensitivity_prior": sens.reset_index().to_dict("records"),
          "sensitivity_cost": sens_c})
assert n_fail == 0

## Ringkasan & keputusan

**Hasil versi ini (v3, Fase 1)**

| | |
|---|---|
| Porsi `D` di frame dinilai | ± 0,48 (sesuai prior; bias 0 memberi ± 0,51 — prior tidak memaksa jauh) |
| Segmen `D` | ± 115 (median ± 18 s) — sebanding dengan jumlah henti permainan yang wajar dalam satu pertandingan |
| Jendela tag set piece yang memuat `D` | 53 / 53 |
| GT publik (4 segmen) | recall ± 0,91, **presisi ± 0,52**, 7 segmen prediksi vs 4 |
| Sensitivitas | jumlah segmen dan angka publik hampir tak berubah untuk prior 0,42–0,54 → prior bukan titik rapuh |

**Yang bekerja:** segmen `D` asli tertangkap (IoU 0,47–0,70); aturan pasti dan bisection bias stabil;
fitur celah bola panjang meliputi ± 41% frame dinilai — selaras dengan dugaan bahwa celah panjang sebagian besar `D`.

**Masalah yang dibawa ke Fase 3.1**

1. **`D` palsu.** Di jendela publik ada 3 segmen `D` lebih banyak dari GT. Setiap `D` palsu memecah possession —
   kesalahan termahal untuk spellF1. Dugaan penyebab: fitur `calm` dibanding norma lokal ± 2 menit, sehingga fase
   permainan lambat (bola diputar di lini belakang) terlihat seperti bola mati. Arah perbaikan: `D` harus didukung
   **kejadian henti** (bola keluar, celah bola panjang, bola diletakkan di titik restart), bukan hanya ketenangan.
2. **Tim restart vs tag: hanya ± 16 dari 30 sepakat.** Di GT publik aturan "tim terdekat di akhir `D`" benar 4/4,
   jadi ketidaksepakatan ini kemungkinan berasal dari **batas akhir `D` yang meleset** (2 s terakhir jatuh di
   permainan hidup) atau dari arti `Team` di tag set piece. Harus diselidiki sebelum restart dipakai sebagai prior kuat.
3. Bobot masih manual. Fase 3.1 menguji alternatif yang dipelajari dari **seluruh match tanpa GT**
   (HMM semi-terawasi dengan jangkar frame pasti).

**Dipakai oleh tahap berikutnya:** `cache/dead.npz` (`d_mask`, `d_score`, `near_team`) dan `cache/restarts_v3.csv`.